In [1]:
import os
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data_tugas = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}

df_tugas4 = pd.DataFrame(data_tugas)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset lokal berhasil dibuat: {df_tugas4.shape[0]} baris.")

# Mengunggah ke HDFS menggunakan variabel dinamis $USER
!hdfs dfs -mkdir -p /user/$USER/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/$USER/tugas4/
print(f"Dataset berhasil diunggah ke HDFS: /user/{os.environ.get('USER', 'kaky')}/tugas4/transaksi_september_2026.csv")

Dataset lokal berhasil dibuat: 1000 baris.
Dataset berhasil diunggah ke HDFS: /user/kaky/tugas4/transaksi_september_2026.csv


In [10]:
import os
import sys
import getpass
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, avg, sum as spark_sum, round as spark_round

# 1. Konfigurasi Environment (Khusus Arch Linux / Safe Isolation)
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas4_BigData_SholahuddinAhmad") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
USERNAME = getpass.getuser()
print(f"SparkSession siap digunakan oleh: {USERNAME}")

SparkSession siap digunakan oleh: kaky


In [12]:
# Jalur HDFS dinamis sesuai user aktif
hdfs_path = f"hdfs://localhost:9000/user/{USERNAME}/tugas4/transaksi_september_2026.csv"

# Membaca dataset CSV dari HDFS
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# 1. Struktur Skema Data
print("=== 1. SKEMA DATA (printSchema) ===")
df.printSchema()

# 2. Jumlah Baris Data
print("\n=== 2. TOTAL BARIS DATA ===")
print("Jumlah baris transaksi:", df.count())

# 3. 10 Baris Pertama
print("\n=== 3. PREVIEW 10 BARIS PERTAMA ===")
df.show(10)

=== 1. SKEMA DATA (printSchema) ===
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)


=== 2. TOTAL BARIS DATA ===
Jumlah baris transaksi: 1000

=== 3. PREVIEW 10 BARIS PERTAMA ===
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|          

In [13]:
# 1. Menghitung jumlah nilai kosong pada kolom rating
jumlah_null = df.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {jumlah_null} baris")

# 2. Menghitung rata-rata rating dari data yang tidak null
rata_rating = df.select(avg("rating")).first()[0]
print(f"Rata-rata rating data valid: {rata_rating:.2f}")

# 3. Menangani missing value dengan imputasi nilai rata-rata (na.fill)
df_clean = df.na.fill({"rating": round(rata_rating, 1)})

# Verifikasi: pastikan sudah tidak ada lagi nilai null
print("Sisa nilai null setelah imputasi:", df_clean.filter(col("rating").isNull()).count())

Jumlah baris dengan rating kosong: 204 baris
Rata-rata rating data valid: 4.15
Sisa nilai null setelah imputasi: 0


> Alasan Logis Pemilihan Metode Imputasi (df.na.fill):

> Terdapat 204 baris data (~20.4%) yang memiliki nilai rating kosong. Jika menggunakan df.na.drop(), kita akan kehilangan seperlima dari total volume transaksi yang berakibat pada distorsi perhitungan finansial (total_pendapatan dan unit_terjual). Oleh karena itu, dipilih metode imputasi nilai rata-rata (df.na.fill) bernilai 4.1 agar volume transaksi tetap utuh dan data rating mendekati distribusi aslinya.

In [14]:
# 1. total_pendapatan = unit_terjual * harga_satuan
# 2. tier_transaksi = 'Besar' jika total_pendapatan > 500.000, selain itu 'Kecil'
df_transformed = df_clean \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

# Tampilkan beberapa baris untuk verifikasi kolom baru
df_transformed.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [16]:
top_kategori = df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc())

top_kategori.show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



In [18]:
top_kota_besar = df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc())

top_kota_besar.show()

+----------+----------------------+
|      kota|jumlah_transaksi_besar|
+----------+----------------------+
|      Solo|                    92|
|  Magelang|                    78|
|   Kebumen|                    78|
|Yogyakarta|                    75|
| Purworejo|                    66|
|  Semarang|                    65|
+----------+----------------------+



In [19]:
rating_metode = df_transformed.groupBy("metode_pembayaran") \
    .agg(spark_round(avg("rating"), 2).alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc())

rating_metode.show()

+-----------------+----------------+
|metode_pembayaran|rata_rata_rating|
+-----------------+----------------+
|              COD|            4.16|
|    Transfer Bank|            4.15|
|         E-Wallet|            4.13|
|     Kartu Kredit|            4.11|
+-----------------+----------------+



In [22]:
# 1. Menyimpan DataFrame ke HDFS
output_hdfs_path = f"hdfs://localhost:9000/user/{USERNAME}/tugas4/hasil_olahan"

# Simpan DataFrame ke HDFS dalam format CSV
df_transformed.write.csv(output_hdfs_path, header=True, mode="overwrite")
print(f"Data berhasil disimpan ke HDFS: {output_hdfs_path}")

# 2. Verifikasi File Partisi di HDFS
!hdfs dfs -ls -h /user/$USER/tugas4/hasil_olahan/

Data berhasil disimpan ke HDFS: hdfs://localhost:9000/user/kaky/tugas4/hasil_olahan
Found 2 items
-rw-r--r--   3 kaky supergroup          0 2026-09-16 22:21 /user/kaky/tugas4/hasil_olahan/_SUCCESS
-rw-r--r--   3 kaky supergroup     95.0 K 2026-09-16 22:21 /user/kaky/tugas4/hasil_olahan/part-00000-921b2f73-03bb-4a60-83b2-da03fc3da40a-c000.csv
